In [ ]:
import redis
import time
import json
from faker import Faker

# Inicializa o Faker para gerar dados de exemplo em português
fake = Faker('pt_BR')

# Garanta que o seu servidor Redis esteja rodando (geralmente em localhost, porta 6379).
# `decode_responses=True` faz com que o Redis retorne strings em vez de bytes, facilitando o uso.
try:
    redis_client = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    # O comando PING é uma  forma de verificar se a conexão foi estabelecida.
    redis_client.ping()
    print("✅ Conexão com o Redis estabelecida com sucesso!")
    redis_client.flushdb()
except redis.exceptions.ConnectionError as e:
    print(f"❌ Falha ao conectar com o Redis: {e}")


 Simulando fonte de dados mais "lenta" (e.g., BD) ...


In [ ]:
def buscar_dados_no_banco(id_usuario: int) -> dict:
    """
    Esta função simula uma consulta lenta a um banco de dados.
    Ela espera 2 segundos e então retorna um dicionário com dados de um usuário.
    """
    print(f"\n⚙️ Consultando o BANCO DE DADOS pelo usuário {id_usuario}... (Isso vai demorar)")
    time.sleep(2)  # Simula a latência de uma consulta complexa ou I/O de disco

    dados_usuario = {
        'id': id_usuario,
        'nome': fake.name(),
        'email': fake.email(),
        'cidade': fake.city(),
        'criado_em': str(fake.past_datetime())
    }
    print("✔️ Dados encontrados no banco!")
    return dados_usuario

In [ ]:
def obter_dados_usuario(id_usuario: int) -> dict:
    """
    Busca os dados de um usuário, implementando a estratégia de cache.
    """
    # É uma boa prática criar um padrão de chave para organizar os dados no Redis.
    chave_cache = f"usuario:{id_usuario}"

    # 1. Tenta buscar os dados primeiro no CACHE (Redis). Esta operação é muito rápida.
    print(f"🔎 Tentando encontrar a chave '{chave_cache}' no Redis...")
    dados_em_cache = redis_client.get(chave_cache)

    # 2. Verifica se houve um CACHE HIT
    if dados_em_cache:
        print(f"✅ CACHE HIT! Dados encontrados no Redis. Retornando instantaneamente.")
        # Os dados são armazenados como uma string JSON, então precisamos convertê-la de volta para um dicionário.
        return json.loads(dados_em_cache)

    # 3. Se não encontrou, é um CACHE MISS
    print(f"❌ CACHE MISS! Chave '{chave_cache}' não encontrada no cache.")

    # Como os dados não estão no cache, buscamos na fonte original (lenta).
    dados_bd = buscar_dados_no_banco(id_usuario)

    # 4. SALVAR NO CACHE para a próxima vez.
    # Convertemos o dicionário para uma string JSON para armazenar no Redis.
    # `ex=30` define um tempo de expiração de 30 segundos (TTL - Time To Live).
    print(f"📥 Salvando novos dados no cache. A chave '{chave_cache}' irá expirar em 120 segundos.")
    redis_client.set(chave_cache, json.dumps(dados_bd), ex=120)

    return dados_bd

In [ ]:
id_alvo = 123

print("\n--- 1ª busca por user=123 ---")
start_time = time.time()
usuario = obter_dados_usuario(id_alvo)
end_time = time.time()

print(f"\nResultado final: {usuario}")
print(f"🕒 Tempo da primeira busca (Cache Miss): {end_time - start_time:.4f} segundos.\n")
print("="*60)

In [ ]:
print("\n--- 2ª busca pelo mesmo usuário (deve ser um Cache Hit) ---")
start_time = time.time()
usuario = obter_dados_usuario(id_alvo)
end_time = time.time()

print(f"\nResultado final: {usuario}")
print(f"🕒 Tempo da segunda busca (Cache Hit): {end_time - start_time:.4f} segundos.")

In [ ]:
redis_client.keys("*")

In [ ]:
redis_client.get("usuario:123")

In [ ]:
redis_client.ttl("usuario:123")